# 01 – Exploration et qualité des données

## Objectif du notebook

Ce notebook a pour objectif de réaliser un premier diagnostic qualité des données utilisées dans le volet B Data Analyst du projet Néovolt Grid+.

Les contrôles réalisés portent sur :
- le volume des fichiers ;
- les colonnes disponibles ;
- les types de données ;
- les valeurs manquantes ;
- les doublons ;
- les valeurs aberrantes ;
- les plages de dates ;
- la cohérence des clés entre les fichiers.

Les fichiers nettoyés seront ensuite générés dans `volet-b-data-analyst/data_cleaned/`.

In [11]:
from pathlib import Path

import pandas as pd
import numpy as np

In [12]:
ROOT_DIR = Path.cwd()

# Si le notebook est exécuté depuis le dossier notebooks, on remonte à la racine du projet
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

DATA_DIR = ROOT_DIR / "donnees"
OUTPUT_DIR = ROOT_DIR / "volet-b-data-analyst" / "data_cleaned"

DATA_DIR, OUTPUT_DIR

(WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/donnees'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-b-data-analyst/data_cleaned'))

In [13]:
files = {
    "actifs_si": "actifs_si.csv",
    "cas_fraude_confirmes": "cas_fraude_confirmes.csv",
    "clients": "clients.csv",
    "compteurs": "compteurs.csv",
    "incidents_reseau": "incidents_reseau.csv",
    "journaux_securite": "journaux_securite.csv",
    "meteo": "meteo.csv",
    "reclamations": "reclamations.csv",
    "releves_consommation": "releves_consommation.csv",
    "releves_horaires": "releves_horaires_echantillon.csv",
}

dataframes = {}

for name, filename in files.items():
    path = DATA_DIR / filename
    if path.exists():
        dataframes[name] = pd.read_csv(path)
        print(f"{name}: chargé avec succès - {dataframes[name].shape[0]} lignes, {dataframes[name].shape[1]} colonnes")
    else:
        print(f"{name}: fichier introuvable -> {path}")

actifs_si: chargé avec succès - 28 lignes, 7 colonnes
cas_fraude_confirmes: chargé avec succès - 24 lignes, 4 colonnes
clients: chargé avec succès - 700 lignes, 7 colonnes
compteurs: chargé avec succès - 700 lignes, 9 colonnes
incidents_reseau: chargé avec succès - 420 lignes, 7 colonnes
journaux_securite: chargé avec succès - 47824 lignes, 6 colonnes
meteo: chargé avec succès - 5848 lignes, 5 colonnes
reclamations: chargé avec succès - 3000 lignes, 6 colonnes
releves_consommation: chargé avec succès - 512986 lignes, 4 colonnes
releves_horaires: chargé avec succès - 21600 lignes, 4 colonnes


In [14]:
summary_rows = []

for name, df in dataframes.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isna().sum().sum()
    
    summary_rows.append({
        "fichier": name,
        "nb_lignes": df.shape[0],
        "nb_colonnes": df.shape[1],
        "doublons_lignes_completes": df.duplicated().sum(),
        "valeurs_manquantes_total": missing_cells,
        "taux_valeurs_manquantes_%": round((missing_cells / total_cells) * 100, 2) if total_cells > 0 else 0,
        "memoire_mo": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

qualite_globale = pd.DataFrame(summary_rows)
qualite_globale = qualite_globale.sort_values(by="nb_lignes", ascending=False)

qualite_globale

,fichier,nb_lignes,nb_colonnes,doublons_lignes_completes,valeurs_manquantes_total,taux_valeurs_manquantes_%,memoire_mo
8,releves_consommation,512986,4,1286,6856,0.33,91.36
5,journaux_securite,47824,6,0,0,0.00,16.70
9,releves_horaires,21600,4,0,0,0.00,4.04
6,meteo,5848,5,0,0,0.00,0.80
7,reclamations,3000,6,0,0,0.00,1.14
2,clients,700,7,0,274,5.59,0.17
3,compteurs,700,9,0,0,0.00,0.32
4,incidents_reseau,420,7,0,0,0.00,0.13
0,actifs_si,28,7,0,0,0.00,0.01
1,cas_fraude_confirmes,24,4,0,0,0.00,0.01


In [15]:
for name, df in dataframes.items():
    print("=" * 80)
    print(f"FICHIER : {name}")
    print("=" * 80)
    
    colonnes_types = pd.DataFrame({
        "colonne": df.columns,
        "type_detecte": df.dtypes.astype(str).values,
        "nb_valeurs_uniques": [df[col].nunique(dropna=True) for col in df.columns],
        "nb_valeurs_manquantes": [df[col].isna().sum() for col in df.columns],
        "taux_manquant_%": [round(df[col].isna().mean() * 100, 2) for col in df.columns]
    })
    
    display(colonnes_types)

FICHIER : actifs_si


,colonne,type_detecte,nb_valeurs_uniques,nb_valeurs_manquantes,taux_manquant_%
0,id_actif,object,28,0,0.0
1,nom,object,28,0,0.0
2,type,object,7,0,0.0
3,criticite,object,4,0,0.0
4,exposition,object,3,0,0.0
5,proprietaire,object,4,0,0.0
6,donnees_sensibles,object,2,0,0.0


FICHIER : cas_fraude_confirmes


,colonne,type_detecte,nb_valeurs_uniques,nb_valeurs_manquantes,taux_manquant_%
0,id_pdl,object,24,0,0.0
1,date_detection,object,23,0,0.0
2,type_fraude,object,3,0,0.0
3,statut,object,1,0,0.0


FICHIER : clients


...

,colonne,type_detecte,nb_valeurs_uniques,nb_valeurs_manquantes,taux_manquant_%
0,id_reclamation,object,3000,0,0.0
1,id_client,object,687,0,0.0
2,date,object,848,0,0.0
3,canal,object,4,0,0.0
4,texte,object,543,0,0.0
5,satisfaction,int64,5,0,0.0


FICHIER : releves_consommation


,colonne,type_detecte,nb_valeurs_uniques,nb_valeurs_manquantes,taux_manquant_%
0,id_pdl,object,700,0,0.00
1,date,object,731,0,0.00
2,consommation_kwh,float64,51767,6856,1.34
3,zone,object,8,0,0.00


FICHIER : releves_horaires


,colonne,type_detecte,nb_valeurs_uniques,nb_valeurs_manquantes,taux_manquant_%
0,id_pdl,object,30,0,0.0
1,horodatage,object,720,0,0.0
2,consommation_kwh,float64,2125,0,0.0
3,zone,object,8,0,0.0


In [16]:
for name, df in dataframes.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    print("=" * 80)
    print(f"VALEURS MANQUANTES : {name}")
    print("=" * 80)
    
    if missing.empty:
        print("Aucune valeur manquante détectée.")
    else:
        missing_report = pd.DataFrame({
            "nb_valeurs_manquantes": missing,
            "taux_manquant_%": round((missing / len(df)) * 100, 2)
        })
        display(missing_report)

VALEURS MANQUANTES : actifs_si
Aucune valeur manquante détectée.
VALEURS MANQUANTES : cas_fraude_confirmes
Aucune valeur manquante détectée.
VALEURS MANQUANTES : clients


,nb_valeurs_manquantes,taux_manquant_%
nb_personnes_foyer,274,39.14


VALEURS MANQUANTES : compteurs
Aucune valeur manquante détectée.
VALEURS MANQUANTES : incidents_reseau
Aucune valeur manquante détectée.
VALEURS MANQUANTES : journaux_securite
Aucune valeur manquante détectée.
VALEURS MANQUANTES : meteo
Aucune valeur manquante détectée.
VALEURS MANQUANTES : reclamations
Aucune valeur manquante détectée.
VALEURS MANQUANTES : releves_consommation


,nb_valeurs_manquantes,taux_manquant_%
consommation_kwh,6856,1.34


VALEURS MANQUANTES : releves_horaires
Aucune valeur manquante détectée.


In [17]:
key_checks = {
    "actifs_si": ["id_actif"],
    "clients": ["id_client"],
    "compteurs": ["id_pdl"],
    "incidents_reseau": ["id_incident"],
    "reclamations": ["id_reclamation"],
    "releves_consommation": ["id_pdl", "date"],
    "releves_horaires": ["id_pdl", "horodatage"],
    "meteo": ["date", "zone"],
    "cas_fraude_confirmes": ["id_pdl", "date_detection"],
}

doublons_metier_rows = []

for name, keys in key_checks.items():
    if name in dataframes:
        df = dataframes[name]
        existing_keys = [key for key in keys if key in df.columns]
        
        if len(existing_keys) == len(keys):
            nb_doublons = df.duplicated(subset=existing_keys).sum()
            doublons_metier_rows.append({
                "fichier": name,
                "cles_controlees": ", ".join(existing_keys),
                "nb_doublons_metier": nb_doublons,
                "taux_doublons_%": round((nb_doublons / len(df)) * 100, 2)
            })
        else:
            doublons_metier_rows.append({
                "fichier": name,
                "cles_controlees": "clé absente ou incomplète",
                "nb_doublons_metier": None,
                "taux_doublons_%": None
            })

doublons_metier = pd.DataFrame(doublons_metier_rows)
doublons_metier

,fichier,cles_controlees,nb_doublons_metier,taux_doublons_%
0,actifs_si,id_actif,0,0.00
1,clients,id_client,0,0.00
2,compteurs,id_pdl,0,0.00
3,incidents_reseau,id_incident,0,0.00
4,reclamations,id_reclamation,0,0.00
5,releves_consommation,"id_pdl, date",1286,0.25
6,releves_horaires,"id_pdl, horodatage",0,0.00
7,meteo,"date, zone",0,0.00
8,cas_fraude_confirmes,"id_pdl, date_detection",0,0.00


In [18]:
date_checks = {
    "clients": ["date_entree"],
    "compteurs": ["date_pose"],
    "incidents_reseau": ["date_debut"],
    "meteo": ["date"],
    "reclamations": ["date"],
    "releves_consommation": ["date"],
    "releves_horaires": ["horodatage"],
    "journaux_securite": ["horodatage"],
    "cas_fraude_confirmes": ["date_detection"],
}

date_rows = []

for name, date_cols in date_checks.items():
    if name in dataframes:
        df = dataframes[name]
        
        for col in date_cols:
            if col in df.columns:
                parsed_dates = pd.to_datetime(df[col], errors="coerce")
                
                date_rows.append({
                    "fichier": name,
                    "colonne_date": col,
                    "nb_lignes": len(df),
                    "dates_invalides": parsed_dates.isna().sum(),
                    "date_min": parsed_dates.min(),
                    "date_max": parsed_dates.max()
                })

controle_dates = pd.DataFrame(date_rows)
controle_dates

,fichier,colonne_date,nb_lignes,dates_invalides,date_min,date_max
0,clients,date_entree,700,0,2009-01-05 00:00:00,2024-12-27 00:00:00
1,compteurs,date_pose,700,0,2016-01-11 00:00:00,2024-12-26 00:00:00
2,incidents_reseau,date_debut,420,0,2024-01-01 13:39:00,2025-12-28 08:48:00
3,meteo,date,5848,0,2024-01-01 00:00:00,2025-12-31 00:00:00
4,reclamations,date,3000,0,2024-01-01 00:00:00,2026-05-29 00:00:00
5,releves_consommation,date,512986,0,2024-01-01 00:00:00,2025-12-31 00:00:00
6,releves_horaires,horodatage,21600,0,2025-08-23 00:00:00,2025-09-21 23:00:00
7,journaux_securite,horodatage,47824,0,2026-03-01 00:23:27,2026-05-29 23:39:49
8,cas_fraude_confirmes,date_detection,24,0,2024-10-22 00:00:00,2025-12-17 00:00:00


In [19]:
releves = dataframes["releves_consommation"].copy()
releves["consommation_kwh"] = pd.to_numeric(releves["consommation_kwh"], errors="coerce")

conso = releves["consommation_kwh"]

q1 = conso.quantile(0.25)
q3 = conso.quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

controle_conso = pd.DataFrame({
    "indicateur": [
        "nb_lignes",
        "valeurs_manquantes",
        "valeurs_negatives",
        "valeurs_nulles",
        "minimum",
        "maximum",
        "moyenne",
        "mediane",
        "q1",
        "q3",
        "borne_basse_iqr",
        "borne_haute_iqr",
        "valeurs_superieures_borne_haute"
    ],
    "valeur": [
        len(conso),
        conso.isna().sum(),
        (conso < 0).sum(),
        (conso == 0).sum(),
        conso.min(),
        conso.max(),
        round(conso.mean(), 2),
        round(conso.median(), 2),
        round(q1, 2),
        round(q3, 2),
        round(borne_basse, 2),
        round(borne_haute, 2),
        (conso > borne_haute).sum()
    ]
})

controle_conso

,indicateur,valeur
0,nb_lignes,512986.00
1,valeurs_manquantes,6856.00
2,valeurs_negatives,1034.00
3,valeurs_nulles,639.00
4,minimum,-1457.60
5,maximum,20766.90
6,moyenne,80.81
7,mediane,14.42
8,q1,8.60
9,q3,44.79


In [20]:
# Top 10 des consommations quotidiennes les plus élevées

top_conso = releves.sort_values(by="consommation_kwh", ascending=False).head(10)
top_conso

,id_pdl,date,consommation_kwh,zone
42596,PDL-000059,2024-03-22,20766.90,Zone-Industrielle
431128,PDL-000589,2024-07-17,19664.82,Centre-Ville
443780,PDL-000606,2025-01-31,16187.47,Centre-Ville
159040,PDL-000218,2024-01-10,16110.45,Rives-Sud
499259,PDL-000682,2024-07-04,15805.50,Zone-Industrielle
338450,PDL-000462,2025-08-11,15650.85,Zone-Industrielle
159047,PDL-000218,2024-01-17,15246.53,Rives-Sud
155027,PDL-000212,2025-01-20,15152.90,Centre-Ville
43196,PDL-000059,2025-11-12,15055.92,Zone-Industrielle
431065,PDL-000589,2024-05-15,14548.49,Centre-Ville
